
# Modelagem de Preditor de Match entre Candidato e Vaga com XGBoost

Este notebook executa todo o pipeline de pré-processamento, treinamento e avaliação de um modelo XGBoost para prever o *match* entre candidatos e vagas de emprego.

Inclui:
- Cálculo de similaridade textual entre currículo e descrição da vaga
- Geração de variáveis auxiliares (features de *match* entre atributos)
- Pré-processamento com `ColumnTransformer`
- Treinamento com `XGBClassifier` e tratamento de desbalanceamento
- Ajuste de *threshold* por grupo de `nível profissional da vaga`


In [110]:

import os
import joblib
import numpy as np
import pandas as pd
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, precision_recall_curve
from xgboost import XGBClassifier


In [111]:

# Carregar dataset balanceado
caminho_dataset = os.path.join("..", "output", "dataset_unificado_balanceado.csv")
df = pd.read_csv(caminho_dataset)
print("Formato do dataset:", df.shape)
df.head()


Formato do dataset: (6386, 22)


,vaga_id,titulo_vaga,cliente,nivel_profissional_vaga,nivel_academico_vaga,nivel_ingles_vaga,nivel_espanhol_vaga,local_vaga,requisitos_vaga,codigo_candidato,...,nivel_ingles_candidato,nivel_espanhol_candidato,conhecimentos_tecnicos,cv_texto,nivel_profissional_candidato,local_candidato,situacao,comentario,recrutador,match
0,3733,Analista Linux e Unix - SR - RE 252407,"Morris, Moran and Dodson",Sênior,Superior,Outro,Outro,São Paulo,Mandatórios Experiência na área de Infraestrut...,22865,...,Desconhecido,Desconhecido,NaN,NaN,Desconhecido,Desconhecido,Contratado pela Decision,Confirmado início hoje,Ana Lívia Moreira,1
1,3395,Analista QA - Pl - RE-264340*,"Morris, Moran and Dodson",Pleno,Superior,Avançado,Outro,São Paulo,Inicio: Imediato Local: Remoto (durante pandem...,22188,...,Desconhecido,Desconhecido,NaN,idade: 35 anos estado civil: solteira principa...,NaN,Desconhecido,Não Aprovado pelo RH,"""O cliente alterou a vaga para inglês fluente ...",Yasmin da Rosa,0
2,8912,Analista de RH Junior,"Morris, Moran and Dodson",Júnior,Superior,Intermediário,Outro,Campinas,Excel avançado ou intermediário; • Facilidade ...,35197,...,Intermediário,Básico,NaN,objetivo tenho como objetivo uma vaga de anali...,NaN,Desconhecido,Inscrito,NaN,Caroline Machado,0
3,4403,Dev open-Analista programador open - FRONT-END...,Mann and Sons,Júnior,Superior,Avançado,Outro,São Paulo,FRONT-END: Must to have Desenvolvimento Angula...,24946,...,Avançado,Avançado,NaN,apinfo: 561.475 22 anos pretensão salarial clt...,NaN,Desconhecido,Contratado como Hunting,NaN,Dra. Luara Siqueira,1
4,3859,Java - 11598644,Nelson-Page,Analista,Superior,Básico,Básico,São Paulo,Java Programming language - Profissionais curs...,22493,...,Desconhecido,Desconhecido,NaN,NaN,Desconhecido,Desconhecido,Encaminhado ao Requisitante,NaN,Carolina Araújo,0


In [112]:

# Limpeza de texto e cálculo de similaridade textual com TF-IDF
df['requisitos_vaga'] = df['requisitos_vaga'].fillna('').str.lower()
df['cv_texto'] = df['cv_texto'].fillna('').str.lower()

todos_textos = pd.concat([df['requisitos_vaga'], df['cv_texto']])
vetorizador_sim = TfidfVectorizer(max_features=300)
vetorizador_sim.fit(todos_textos)

req_matrix = vetorizador_sim.transform(df['requisitos_vaga'])
cv_matrix = vetorizador_sim.transform(df['cv_texto'])
df['sim_textual'] = np.array(req_matrix.multiply(cv_matrix).sum(axis=1)).ravel()

df['sim_textual'] = df['sim_textual'] * 0.3  # Reduz peso da feature


In [113]:

# Features de comparação entre atributos do candidato e da vaga
df['match_nivel'] = (df['nivel_profissional_vaga'] == df['nivel_profissional_candidato']).astype(int)
df['match_ingles'] = (df['nivel_ingles_vaga'] == df['nivel_ingles_candidato']).astype(int)
df['match_profissional'] = (df['nivel_profissional_vaga'] == df['nivel_profissional_candidato']).astype(int)
df['match_espanhol'] = (df['nivel_espanhol_vaga'] == df['nivel_espanhol_candidato']).astype(int)
df['match_local'] = (df['local_vaga'] == df['local_candidato']).astype(int)
df['match_academico'] = (df['nivel_academico_vaga'] == df['nivel_academico_candidato']).astype(int)


In [114]:

# Remoção de colunas que causariam vazamento de informação
colunas_vazamento = ['situacao', 'comentario', 'recrutador']
X = df.drop(columns=['match'] + colunas_vazamento)
y = df['match']

cat_cols = [
    'titulo_vaga', 'nivel_profissional_vaga', 'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga',
    'nivel_academico_candidato', 'nivel_ingles_candidato', 'nivel_espanhol_candidato',
    'nivel_profissional_candidato', 'local_candidato', 'cliente', 'local_vaga'
]

num_cols = [
    'sim_textual', 'match_nivel', 'match_ingles', 'match_profissional',
    'match_espanhol', 'match_local', 'match_academico'
]

# Subset final de colunas
colunas_utilizadas = [col for col in cat_cols + num_cols + ['requisitos_vaga', 'cv_texto'] if col in X.columns]
X = X[colunas_utilizadas]


In [115]:

# Pipelines de transformação
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

text_pipeline = TfidfVectorizer(max_features=30)

cat_cols_validos = [col for col in cat_cols if col in X.columns]
num_cols_validos = [col for col in num_cols if col in X.columns]

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols_validos),
    ('cat', cat_pipeline, cat_cols_validos),
    ('req_text', text_pipeline, 'requisitos_vaga'),
    ('cv_text', text_pipeline, 'cv_texto')
], sparse_threshold=1.0)


In [116]:

X_train_raw, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Peso de classe para balancear
total_pos = y_train.sum()
scale_pos_weight = (len(y_train) - total_pos) / total_pos
print("scale_pos_weight:", scale_pos_weight)


scale_pos_weight: 1.0


In [117]:

# Transformar os dados
X_train_transf = preprocessor.fit_transform(X_train_raw)
X_test_transf = preprocessor.transform(X_test)

# Modelo XGBoost
modelo = XGBClassifier(
    n_estimators=100,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    early_stopping_rounds=10,
    random_state=42
)
modelo.fit(X_train_transf, y_train, eval_set=[(X_test_transf, y_test)], verbose=False)


c:\Users\maiar\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\callback.py:386: UserWarning: [13:32:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=10,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [118]:

from sklearn.metrics import classification_report

y_pred_default = modelo.predict(X_test_transf)
print("Avaliação com threshold padrão:")
print(classification_report(y_test, y_pred_default, digits=3))


Avaliação com threshold padrão:
              precision    recall  f1-score   support

           0      0.695     0.831     0.757       639
           1      0.790     0.635     0.704       639

    accuracy                          0.733      1278
   macro avg      0.742     0.733     0.731      1278
weighted avg      0.742     0.733     0.731      1278



In [119]:

import logging
from datetime import datetime

# Configurar logging
logging.basicConfig(
    filename="log_inferencias.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


In [120]:
import logging
from datetime import datetime
from sklearn.metrics import classification_report

# Configurar logging
logging.basicConfig(
    filename="log_inferencias.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Gerar relatório
relatorio = classification_report(y_test, y_pred_fixo, digits=3)

# Logging da avaliação
logging.info("=== Avaliação com threshold fixo ===")
logging.info("Threshold utilizado: %.2f", THRESHOLD_PADRAO)
logging.info("Total de exemplos: %d", len(y_pred_fixo))
logging.info("Número de positivos previstos: %d", y_pred_fixo.sum())
logging.info("Número de negativos previstos: %d", (y_pred_fixo == 0).sum())
logging.info("\n%s", relatorio)


In [121]:

# Ajuste de threshold por grupo de nível profissional da vaga
y_probs = modelo.predict_proba(X_test_transf)[:, 1]
X_test_com_probs = X_test.copy()
X_test_com_probs['prob'] = y_probs
X_test_com_probs['true'] = y_test.values

thresholds_dinamicos = {
    "Júnior": 0.60,
    "Sênior": 0.54,
    "Analista": 0.43,
    "Assistente": 0.60,
    "Pleno": 0.46,
    "Especialista": 0.49
}

for grupo in X_test_com_probs['nivel_profissional_vaga'].unique():
    grupo_mask = X_test_com_probs['nivel_profissional_vaga'] == grupo
    y_true_grupo = X_test_com_probs.loc[grupo_mask, 'true']
    y_probs_grupo = X_test_com_probs.loc[grupo_mask, 'prob']

    if len(y_true_grupo) < 30:
        continue

    precs, recs, thrs = precision_recall_curve(y_true_grupo, y_probs_grupo)

    valid_idxs = [
        i for i in range(len(thrs))
        if recs[i] >= 0.6 and precs[i] >= 0.25
    ]

    if valid_idxs:
        best_idx = max(valid_idxs, key=lambda i: precs[i])
        thresholds_dinamicos[grupo] = float(thrs[best_idx])
    else:
        thresholds_dinamicos[grupo] = 0.5

# Logging da avaliação
logging.info("=== Avaliação com threshold fixo ===")
logging.info("Threshold utilizado: %.2f", THRESHOLD_PADRAO)
logging.info("Total de exemplos: %d", len(y_pred_fixo))
logging.info("Número de positivos previstos: %d", y_pred_fixo.sum())
logging.info("Número de negativos previstos: %d", (y_pred_fixo == 0).sum())
logging.info("\n" + relatorio)


In [122]:
# Threshold fixo
THRESHOLD_PADRAO = 0.5

# Predição com threshold fixo
y_pred_fixo = (X_test_com_probs["prob"] >= THRESHOLD_PADRAO).astype(int)

print("Avaliação com threshold fixo:")
print(classification_report(y_test, y_pred_fixo, digits=3))

# Salvar artefatos
output_dir = os.path.join("..", "output")
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo, os.path.join(output_dir, "modelo_match_xgb.joblib"))
joblib.dump(vetorizador_sim, os.path.join(output_dir, "vetorizador_sim_textual.joblib"))
joblib.dump(preprocessor, os.path.join(output_dir, "preprocessador_xgb.joblib"))

print("Pipeline completo. Artefatos salvos.")

Avaliação com threshold fixo:
              precision    recall  f1-score   support

           0      0.695     0.831     0.757       639
           1      0.790     0.635     0.704       639

    accuracy                          0.733      1278
   macro avg      0.742     0.733     0.731      1278
weighted avg      0.742     0.733     0.731      1278

Pipeline completo. Artefatos salvos.
